## A notebook to create a bar graph of CTs inside AS

## Install and import libraries

In [74]:

%pip install pandas

import pandas as pd
import requests
from  io import StringIO
from pprint import pprint

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Global settings

In [75]:
hra_pop_version = 'v0.12.0'
branch = 'v0.12.0'

output_folder = 'output/ctann-tree'

## Manually extract CTs from new CTann crosswalks

In [76]:
crosswalk_azimuth = pd.read_csv('data/azimuth_v1.2_DRAFT.csv', skiprows=10)
crosswalk_azimuth

,Organ_Level,Organ_ID,Annotation_Label,Annotation_Label_ID,CL_Label,CL_ID,CL_Match
0,Heart_L2,UBERON:0000948,Adipocyte,AZ:0000001,adipocyte,CL:0000136,skos:exactMatch
1,Heart_L2,UBERON:0000948,Arterial Endothelial,AZ:0000002,endothelial cell of artery,CL:1000413,skos:exactMatch
2,Heart_L2,UBERON:0000948,Atrial Cardiomyocyte,AZ:0000003,regular atrial cardiac myocyte,CL:0002129,skos:exactMatch
3,Heart_L2,UBERON:0000948,B,AZ:0000004,B cell,CL:0000236,skos:exactMatch
4,Heart_L2,UBERON:0000948,Capillary Endothelial,AZ:0000005,capillary endothelial cell,CL:0002144,skos:exactMatch
...,...,...,...,...,...,...,...
759,Kidney,UBERON:0002113,Peritubular Capilary Endothelial,NaN,peritubular capillary endothelial cell,CL:1001033,skos:exactMatch
760,Bone_marrow,UBERON:0002371,CD8 Effector_1,NaN,"effector CD8-positive, alpha-beta T cell:1",CL:0001050,skos:narrowMatch
761,Bone_marrow,UBERON:0002371,CD8 Effector_2,NaN,"effector CD8-positive, alpha-beta T cell:2",CL:0001050,skos:narrowMatch
762,Bone_marrow,UBERON:0002371,CD8 Effector_3,NaN,"effector CD8-positive, alpha-beta T cell:3",CL:0001050,skos:narrowMatch


In [77]:
crosswalk_celltypist = pd.read_csv('data/celltypist_v1.1_DRAFT.csv', skiprows=10)
crosswalk_celltypist

,Organ_Level,Organ_ID,Annotation_Label,Annotation_Label_ID,CL_Label,CL_ID,CL_Match
0,blood_L1,UBERON:0000178,Age-associated B cells,CT:0000001,B cell:age-associated,CL:0000236,skos:narrowMatch
1,blood_L1,UBERON:0000178,C1 non-classical monocytes,CT:0000002,non-classical monocyte:C1,CL:0000875,skos:narrowMatch
2,blood_L1,UBERON:0000178,CD16+ NK cells,CT:0000003,"CD16-positive, CD56-dim natural killer cell, h...",CL:0000939,skos:exactMatch
3,blood_L1,UBERON:0000178,CD16- NK cells,CT:0000004,"CD16-negative, CD56-bright natural killer cell...",CL:0000938,skos:exactMatch
4,blood_L1,UBERON:0000178,Classical monocytes,CT:0000005,classical monocyte,CL:0000860,skos:exactMatch
...,...,...,...,...,...,...,...
888,Small_Intestine,UBERON:0002108,myofibroblast,NaN,myofibroblast cell,CL:0000186,skos:exactMatch
889,Small_Intestine,UBERON:0002108,myofibroblast (RSPO2+),NaN,myofibroblast cell:RSPO2+,CL:0000186,skos:narrowMatch
890,Small_Intestine,UBERON:0002108,pDC,NaN,plasmacytoid dendritic cell,CL:0000784,skos:exactMatch
891,Small_Intestine,UBERON:0002108,venous capillary,NaN,pre-venule capillary cell,CL:4047030,skos:exactMatch


In [78]:
crosswalk_popv = pd.read_csv('data/popv_v1.2_DRAFT.csv', skiprows=10)
crosswalk_popv

,Organ_Level,Organ_ID,Annotation_Label,Annotation_Label_ID,CL_Label,CL_ID,CL_Match
0,blood,UBERON:0000178,CD141-positive myeloid dendritic cell,PV:0000001,CD141-positive myeloid dendritic cell,CL:0002394,skos:exactMatch
1,blood,UBERON:0000178,"CD4-positive, alpha-beta memory T cell",PV:0000002,"CD4-positive, alpha-beta memory T cell",CL:0000897,skos:exactMatch
2,blood,UBERON:0000178,"CD8-positive, alpha-beta T cell",PV:0000003,"CD8-positive, alpha-beta T cell",CL:0000625,skos:exactMatch
3,blood,UBERON:0000178,"CD8-positive, alpha-beta cytokine secreting ef...",PV:0000004,"CD8-positive, alpha-beta cytokine secreting ef...",CL:0000908,skos:exactMatch
4,blood,UBERON:0000178,T cell,PV:0000005,T cell,CL:0000084,skos:exactMatch
...,...,...,...,...,...,...,...
443,prostate gland,UBERON:0002367,bronchial epithelial cell,NaN,epithelial cell,CL:0000066,skos:narrowMatch
444,thymus,UBERON:0002370,"CD4-positive, CD25-positive, alpha-beta regula...",NaN,"CD4-positive, CD25-positive, alpha-beta regula...",CL:0000792,skos:exactMatch
445,thymus,UBERON:0002370,"CD4-positive, alpha-beta T cell",NaN,"CD4-positive, alpha-beta T cell",CL:0000624,skos:exactMatch
446,bone marrow,UBERON:0002371,"B cell, CD19-positive",NaN,"B cell, CD19-positive",CL:0001201,skos:exactMatch


In [79]:
# extract CL IDs and labels with tool
extract = ['CL_ID', 'CL_Label']

# Extract the columns from each DataFrame
az_selected = crosswalk_azimuth[extract].assign(tool='azimuth')
ct_selected = crosswalk_celltypist[extract].assign(tool='celltypist')
popv_selected = crosswalk_popv[extract].assign(tool='popv')

# Concatenate them into one DataFrame
combined_df = pd.concat([az_selected, ct_selected, popv_selected], ignore_index=True)

combined_df

,CL_ID,CL_Label,tool
0,CL:0000136,adipocyte,azimuth
1,CL:1000413,endothelial cell of artery,azimuth
2,CL:0002129,regular atrial cardiac myocyte,azimuth
3,CL:0000236,B cell,azimuth
4,CL:0002144,capillary endothelial cell,azimuth
...,...,...,...
2100,CL:0000066,epithelial cell,popv
2101,CL:0000792,"CD4-positive, CD25-positive, alpha-beta regula...",popv
2102,CL:0000624,"CD4-positive, alpha-beta T cell",popv
2103,CL:0001201,"B cell, CD19-positive",popv


In [80]:
# reformat for SPARQL at https://api.triplydb.com/s/JTDOCdAFS
parts = []

for index, row in combined_df.iterrows():
  parts.append(f'(CL:{row['CL_ID'].split(":")[-1]})')

unique = set(parts)
formatted_cells = ' '.join(unique)
formatted_cells

'(CL:0000097) (CL:0002350) (CL:0002590) (CL:0009084) (CL:0000577) (CL:1000768) (CL:0000236) (CL:0002275) (CL:0000910) (CL:0000583) (CL:4030016) (CL:0009062) (CL:0019032) (CL:0002370) (CL:0009017) (CL:0000808) (CL:0008001) (CL:0000985) (CL:0009094) (CL:0000624) (CL:0002543) (CL:0000785) (CL:0002598) (CL:0009086) (CL:0000065) (CL:0000677) (CL:1001603) (CL:4033044) (CL:4047003) (CL:0000764) (CL:1001005) (CL:4033069) (CL:0000935) (CL:0011026) (CL:0000573) (CL:0000936) (CL:0000066) (CL:0002480) (CL:0002280) (CL:0002425) (CL:0000940) (CL:0002303) (CL:0000188) (CL:0000669) (CL:4030012) (CL:1001109) (CL:4023011) (CL:0000987) (CL:1000436) (CL:0000129) (CL:0000448) (CL:0000816) (CL:1000452) (CL:4033022) (CL:0000653) (CL:0000860) (CL:0002153) (CL:0009100) (CL:0002503) (CL:1001569) (CL:0000843) (CL:4030011) (CL:1001033) (CL:0009105) (CL:1000223) (CL:4052029) (CL:0000865) (CL:0000492) (CL:0000171) (CL:4023018) (CL:0009080) (CL:0000287) (CL:0002366) (CL:0000189) (CL:4033048) (CL:0002340) (CL:4023043

## Get updated CT level mapping via query with manually extracted CTs from new CTann crosswalks

In [81]:
with open('data/ad_hoc_cell_type_level_mapping_query.rq', 'r') as f:
    query = f.read()
    
# define endpoint
url = "https://lod.humanatlas.io/sparql"

query = query.replace("#L4_CELLS", formatted_cells)

# define parameters
params = {
    "query": query,
}

# set header
headers = {
    "Accept": "text/csv",
    
}

# Send the GET request
response = requests.post(url, headers=headers, data=params)

# convert text to file-like object
csv_data = StringIO(response.text)

# concert to DataFrame
yasgui_cell_types_level_mapping = pd.read_csv(csv_data)
yasgui_cell_types_level_mapping

,cell_label,cell_id,level_1_cell_id,level_1_cell_label
0,cell,http://purl.obolibrary.org/obo/CL_0000000,http://purl.obolibrary.org/obo/CL_0000000,unknown cell
1,neuroblast (sensu Vertebrata),http://purl.obolibrary.org/obo/CL_0000031,http://purl.obolibrary.org/obo/CL_0000000,unknown cell
2,stem cell,http://purl.obolibrary.org/obo/CL_0000034,http://purl.obolibrary.org/obo/CL_0000034,stem cell
3,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000034,stem cell
4,erythroid progenitor cell,http://purl.obolibrary.org/obo/CL_0000038,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
...,...,...,...,...
450,fetal artery endothelial cell,http://purl.obolibrary.org/obo/CL_4047006,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
451,immature pericyte,http://purl.obolibrary.org/obo/CL_4047007,http://purl.obolibrary.org/obo/CL_0000000,unknown cell
452,fetal vein endothelial cell,http://purl.obolibrary.org/obo/CL_4047011,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
453,angiogenic pericyte,http://purl.obolibrary.org/obo/CL_4047012,http://purl.obolibrary.org/obo/CL_0000000,unknown cell


In [82]:
# yasgui_cell_types_level_mapping['level_1_cell_id'] = yasgui_cell_types_level_mapping['level_1_cell_id'].apply(
#     lambda id: id.split('/')[-1].replace('_', ':'))
yasgui_cell_types_level_mapping

,cell_label,cell_id,level_1_cell_id,level_1_cell_label
0,cell,http://purl.obolibrary.org/obo/CL_0000000,http://purl.obolibrary.org/obo/CL_0000000,unknown cell
1,neuroblast (sensu Vertebrata),http://purl.obolibrary.org/obo/CL_0000031,http://purl.obolibrary.org/obo/CL_0000000,unknown cell
2,stem cell,http://purl.obolibrary.org/obo/CL_0000034,http://purl.obolibrary.org/obo/CL_0000034,stem cell
3,hematopoietic stem cell,http://purl.obolibrary.org/obo/CL_0000037,http://purl.obolibrary.org/obo/CL_0000034,stem cell
4,erythroid progenitor cell,http://purl.obolibrary.org/obo/CL_0000038,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
...,...,...,...,...
450,fetal artery endothelial cell,http://purl.obolibrary.org/obo/CL_4047006,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
451,immature pericyte,http://purl.obolibrary.org/obo/CL_4047007,http://purl.obolibrary.org/obo/CL_0000000,unknown cell
452,fetal vein endothelial cell,http://purl.obolibrary.org/obo/CL_4047011,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
453,angiogenic pericyte,http://purl.obolibrary.org/obo/CL_4047012,http://purl.obolibrary.org/obo/CL_0000000,unknown cell


## Join `df` containing `ctann` CTs with new cell-mapping

In [83]:
df_temp = combined_df

# remove ASCTB TEMP
df_temp = df_temp[~df_temp['cell_id'].str.contains('ASCT', na=False)]
df_temp



KeyError: 'cell_id'

In [ ]:
# adjust cell_id column
# df_temp['cell_id'] = df_temp['cell_id'].apply(lambda id: id.split('/')[-1].replace('_', ':'))
df_temp

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count,level_1_cell_id,level_1_cell_label
0,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002079,ductal,15.312,0.522523,1,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell
1,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002064,acinar,8.640,0.294840,1,http://purl.obolibrary.org/obo/CL_0000152,exocrine cell
2,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000115,endothelial,3.864,0.131859,1,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
3,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000738,immune,1.464,0.049959,1,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
4,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002410,activated_stellate,0.024,0.000819,1,http://purl.obolibrary.org/obo/CL_0000499,stromal cell
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9372,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000115,Endothelial,40223.460,0.064846,1,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
9374,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033038,CD4+ T Cell,16776.624,0.027046,1,No higher-level CT,No higher-level CT
9375,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000097,Mast Cell,15322.464,0.024702,1,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
9376,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033039,CD8+ T Cell,3691.176,0.005951,1,No higher-level CT,No higher-level CT


In [ ]:
# Merge look-up df with df
df_temp = combined_df.merge(
    yasgui_cell_types_level_mapping[['cell_id', 'level_1_cell_id', 'level_1_cell_label']],
    left_on='cell_id',  # Column in main df
    right_on='cell_id',  # Column in lookup df
    how='left'      # Keep all rows from main df
)

df_temp

KeyError: 'cell_id'

In [ ]:
# handle missing values
df_temp['level_1_cell_id'] = df_temp['level_1_cell_id'].fillna(
    'No higher-level CT')
df_temp['level_1_cell_label'] = df_temp['level_1_cell_label'].fillna(
    'No higher-level CT')

KeyError: 'level_1_cell_id'

In [ ]:
df = df_temp
df

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count,level_1_cell_id,level_1_cell_label
0,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002079,ductal,15.312,0.522523,1,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell
1,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002064,acinar,8.640,0.294840,1,http://purl.obolibrary.org/obo/CL_0000152,exocrine cell
2,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000115,endothelial,3.864,0.131859,1,http://purl.obolibrary.org/obo/CL_0000115,endothelial cell
3,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000738,immune,1.464,0.049959,1,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
4,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002410,activated_stellate,0.024,0.000819,1,http://purl.obolibrary.org/obo/CL_0000499,stromal cell
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9375,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000097,Mast Cell,15322.464,0.024702,1,http://purl.obolibrary.org/obo/CL_0000988,hematopoietic cell
9376,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033039,CD8+ T Cell,3691.176,0.005951,1,No higher-level CT,No higher-level CT
9377,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_lymphatic-endo...,Lymphatic Endothelial (and some immune cells),1753.956,0.002828,1,No higher-level CT,No higher-level CT
9378,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_basal-epitheli...,Basal Epithelial Cell,970.104,0.001564,1,No higher-level CT,No higher-level CT


In [ ]:
# keep unique combinations of tool, cell_id, cell_label, level_1_cell_id, and level_1_cell_label
df_unique = df.drop_duplicates(subset=['tool', 'cell_id', 'cell_label', 'level_1_cell_id', 'level_1_cell_label'])

# adjust format for ontology ID
for col in ['cell_id', 'level_1_cell_id']:
  df_unique[col] = df[col].apply(lambda x: x.split('/')[-1].replace('_', ':') if isinstance(x, str) else x)

df_unique

C:\Users\abueckle\AppData\Local\Temp\1\ipykernel_42604\4200318101.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique[col] = df[col].apply(lambda x: x.split('/')[-1].replace('_', ':') if isinstance(x, str) else x)


,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count,level_1_cell_id,level_1_cell_label
0,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,CL:0002079,ductal,15.312,0.522523,1,CL:0000066,epithelial cell
1,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,CL:0002064,acinar,8.640,0.294840,1,CL:0000152,exocrine cell
2,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,CL:0000115,endothelial,3.864,0.131859,1,CL:0000115,endothelial cell
3,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,CL:0000738,immune,1.464,0.049959,1,CL:0000988,hematopoietic cell
4,pancreas,http://purl.obolibrary.org/obo/UBERON_0001069,head of pancreas,Female,azimuth,sc_transcriptomics,CL:0002410,activated_stellate,0.024,0.000819,1,CL:0000499,stromal cell
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9368,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,CL:0002062,Type 1 Alveolar Epithelial Cell,292533.168,0.471605,1,CL:0000066,epithelial cell
9369,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,ASCTB-TEMP:mpo-,MPO+,93194.724,0.150243,1,No higher-level CT,No higher-level CT
9375,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,CL:0000097,Mast Cell,15322.464,0.024702,1,CL:0000988,hematopoietic cell
9377,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,ASCTB-TEMP:lymphatic-endothelial-and-some-immu...,Lymphatic Endothelial (and some immune cells),1753.956,0.002828,1,No higher-level CT,No higher-level CT


## Load and enrich SLIM for vis in ASCT+B Reporter, join with `df`

In [ ]:
# sheet_url = 'https://docs.google.com/spreadsheets/d/1JZE9BprxatUUopN25P1G16flGXvbuRipuddRFREkN4A/edit?gid=0#gid=0'
sheet = pd.read_csv('data/SLIM hierarchy.csv', skiprows=2)
df_slim = sheet.fillna("")
df_slim

,AS/1,AS/1/LABEL,AS/1/ID,AS/2,AS/2/LABEL,AS/2/ID,AS/3,AS/3/LABEL,AS/3/ID
0,cell,cell,CL:0000000,connective tissue cell,connective tissue cell,CL:0002320,adipocyte,adipocyte,CL:0000136
1,cell,cell,CL:0000000,melanocyte,melanocyte,CL:0000148,,,
2,cell,cell,CL:0000000,hematopoietic cell,hematopoietic cell,CL:0000988,monocyte,monocyte,CL:0000576
3,cell,cell,CL:0000000,,,,exocrine cell,exocrine cell,CL:0000152
4,cell,cell,CL:0000000,extraembryonic cell,extraembryonic cell,CL:0000349,,,
5,cell,cell,CL:0000000,hematopoietic cell,hematopoietic cell,CL:0000988,,,
6,cell,cell,CL:0000000,germ line cell,germ line cell,CL:0000039,,,
7,cell,cell,CL:0000000,bone cell,bone cell,CL:0001035,,,
8,cell,cell,CL:0000000,hematopoietic cell,hematopoietic cell,CL:0000988,blood cell,blood cell,CL:0000081
9,cell,cell,CL:0000000,neural cell,neural cell,CL:0002319,glial cell,glial cell,CL:0000125


In [ ]:
# Make a copy of df_slim to preserve the original
df_slim_result = df_slim.copy()

# Ensure key columns are treated as strings for safe matching
df_slim['AS/2/ID'] = df_slim['AS/2/ID'].astype(str)
df_slim['AS/3/ID'] = df_slim['AS/3/ID'].astype(str)
df_unique['level_1_cell_id'] = df_unique['level_1_cell_id'].astype(str)

for _, row in df_unique.iterrows():
    level_1_cell_id = str(row['level_1_cell_id'])
    # Replace with your actual column name
    level_1_cell_label = row['level_1_cell_label']

    # Then use `cell_id` as before to match in df_slim
    mask = (df_slim['AS/2/ID'] == level_1_cell_id) & (df_slim['AS/3/ID'] == '')
    matching_rows = df_slim[mask].copy()

    if matching_rows.empty:
        mask = df_slim['AS/3/ID'] == level_1_cell_id
        matching_rows = df_slim[mask].copy()

    if not matching_rows.empty:
        matching_rows['AS/4'] = str(row['cell_label'])
        matching_rows['AS/4/LABEL'] = str(row['cell_label'])
        matching_rows['AS/4/ID'] = str(row['cell_id'])
        df_slim_result = pd.concat([df_slim_result, matching_rows], ignore_index=True)

# Remove original rows in df_slim if there was at least one match
# Step 1: Identify matched cell IDs
matched_ids = set(df_slim_result['AS/4/ID'].dropna().astype(str))

# Step 2: Remove original rows (no AS/4/ID) that matched either AS/2/ID or AS/3/ID
df_slim_result = df_slim_result[
    ~(
        df_slim_result['AS/4/ID'].isna() & (
            df_slim_result['AS/2/ID'].isin(matched_ids) |
            df_slim_result['AS/3/ID'].isin(matched_ids)
        )
    )
]

df_slim_result = df_slim_result.sort_values(by=['AS/4/LABEL'])

C:\Users\abueckle\AppData\Local\Temp\1\ipykernel_42604\3547330226.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique['level_1_cell_id'] = df_unique['level_1_cell_id'].astype(str)


## Export

In [ ]:
df_slim_result.to_csv('output/ctann_tree.csv', index=False)
# put on Google Sheets: https://docs.google.com/spreadsheets/d/1ISKJOktR6pl6uUNhLdV6xcXMZAQ3KfEgf8BMp9Jjs9s/edit?gid=1266919613#gid=1266919613